In [ ]:
import glob
import os
import shutil
from datetime import datetime

# Core data manipulation and numerical libraries
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Machine Learning metrics and hyperparameter tuning utilities
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import ParameterGrid

# Deep Learning framework (PyTorch) modules
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

In [ ]:
# Define the directory path where trained model checkpoints will be stored
base_dir = "your path"

# Define the directory path containing the raw and processed datasets
folder_path = "data\std_data"

In [ ]:
param_grid = {
    # Pipeline configuration
    "resample_minute": ["30T"],
    "random_seed": [168],
    "window_size": [30],

    # Model configuration
    "dropout_rate": [0.3],
    "CNN_kernel_size": [3],
    "CNN_1st_channel": [64],
    "CNN_2st_channel": [128],
    "Transformer_encoder_num_layers": [6],
    "LSTM_hidden_size": [128],
    "LSTM_num_layers": [6],
    "LSTM_bidirectional": [True],
    
    # Optimization configuration
    "learning_rate": [0.0001],
    "patience": [100],
    "num_epochs": [10000],
    "batch_size": [128],
}

# Generate hyperparameter combinations for grid search
grid = list(ParameterGrid(param_grid))

In [ ]:
# Input features for model training (44 features)
features = [
    "F_open",
    "F_high",
    "F_low",
    "F_close",
    "F_vol",
    "TX",
    "TX_volatility",
    "ATMS_interpolated",
    "remaining_time",
    "ATMS_avg",
    "ATMS_pos_std",
    "ATMS_neg_std",
    "ATMS_upbond",
    "ATMS_downbond",
    "ATMS_pos_std_EMA",
    "ATMS_neg_std_EMA",
    "ATMS_Nstd",
    "TX_RV_30",
    "TX_RV_15",
    "TX_RV_10",
    "TX_RV_5",
    "TX_pct_change_30",
    "TX_pct_change_15",
    "TX_pct_change_10",
    "TX_pct_change_5",
    "ATMS_RV_30",
    "ATMS_RV_15",
    "ATMS_RV_10",
    "ATMS_RV_5",
    "ATMS_pct_change_30",
    "ATMS_pct_change_15",
    "ATMS_pct_change_10",
    "ATMS_pct_change_5",
    "F_range",
    "F_body",
    "F_vol_ma_30",
    "F_vol_ma_15",
    "F_vol_ma_10",
    "F_vol_ma_5",
    "F_vol_pct_change_30",
    "F_vol_pct_change_15",
    "F_vol_pct_change_10",
    "F_vol_pct_change_5",
    "time_diff",
]

# Target variable for prediction
target = "next_ATMS"

# Raw dataset base features to retain
cols_to_keep = [
    "F_open",
    "F_high",
    "F_low",
    "F_close",
    "F_vol",
    "TX",
    "TX_volatility",
    "ATMS_interpolated",
    "remaining_time",
    "ATMS_avg",
    "ATMS_pos_std",
    "ATMS_neg_std",
    "ATMS_upbond",
    "ATMS_downbond",
    "ATMS_pos_std_EMA",
    "ATMS_neg_std_EMA",
    "ATMS_Nstd",
]

In [ ]:
class FusionNet(nn.Module):
    def __init__(self, window_size, feature_dim, dropout_rate):
        super(FusionNet, self).__init__()

        # ---- LSTM Preprocessing ----
        self.lstm = nn.LSTM(
            input_size=feature_dim,
            hidden_size=LSTM_hidden_size,
            num_layers=LSTM_num_layers,
            bidirectional=LSTM_bidirectional,
            batch_first=True
        )
        self.lstm_output_dim = LSTM_hidden_size * (2 if LSTM_bidirectional else 1)

        # ---- CNN Branch ----
        self.cnn_branch = nn.Sequential(
            nn.Conv1d(in_channels=self.lstm_output_dim, out_channels=CNN_1st_channel, kernel_size=CNN_kernel_size, padding=1),
            nn.ReLU(),
            nn.Conv1d(in_channels=CNN_1st_channel, out_channels=CNN_2st_channel, kernel_size=CNN_kernel_size, padding=1),
            nn.ReLU()
        )

        # ---- Attention Branch ----
        encoder_layer = nn.TransformerEncoderLayer(d_model=self.lstm_output_dim, nhead=32, dropout=dropout_rate)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=Transformer_encoder_num_layers)

        # ---- Final Output Layer ----
        merged_dim = CNN_2st_channel + self.lstm_output_dim
        self.output_layer = nn.Linear(merged_dim, 1)

    def forward(self, x):
        batch_size, seq_len, feature_dim = x.size()

        # ---- LSTM Preprocessing ----
        lstm_out, _ = self.lstm(x)  # [batch, time, hidden]
        # Keep full time series sequence (do not take only the last layer)

        # ---- CNN Branch ----
        x_cnn = lstm_out.permute(0, 2, 1)  # [batch, channels, time]
        cnn_out = self.cnn_branch(x_cnn)  # [batch, channels, time]
        cnn_out = cnn_out.permute(0, 2, 1)  # [batch, time, channels]

        # ---- Transformer Branch ----
        x_attn = lstm_out.permute(1, 0, 2)  # [time, batch, feature_dim]
        attn_out = self.transformer_encoder(x_attn)  # [time, batch, feature_dim]
        attn_out = attn_out.permute(1, 0, 2)  # [batch, time, feature_dim]

        # ---- Concatenate CNN + Attention Outputs ----
        merged = torch.cat([cnn_out, attn_out], dim=2)  # [batch, time, merged_dim]
        final_hidden = merged[:, -1, :]

        # ---- Output ----
        out = self.output_layer(final_hidden)  # [batch, 1]
        return out

In [ ]:
class LossFunction(nn.Module):
    def __init__(self, epsilon=1e-6, scale=1.2):
        super().__init__()
        self.mse = nn.MSELoss()
        self.epsilon = epsilon
        self.scale = scale

    def forward(self, outputs, targets, base_atms):
        # Calculate standard Mean Squared Error loss
        mse_loss = self.mse(outputs, targets)
        
        # Calculate changes relative to the base ATMS value
        pred_diff = outputs - base_atms
        true_diff = targets - base_atms
        
        # Determine if the predicted direction matches the true direction
        direction_correct = torch.eq(torch.sign(pred_diff), torch.sign(true_diff)).float()
        
        # Calculate directional accuracy (win rate) within the batch
        win_rate = torch.mean(direction_correct)
        
        # Apply logarithmic penalty for incorrect directional predictions
        penalty = -torch.log(win_rate**2 + self.epsilon)
        
        # Return total loss combining magnitude and directional penalties
        return mse_loss * (1 + self.scale * penalty)

In [ ]:
class TimeSeriesDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.from_numpy(X).float()
        self.y = torch.from_numpy(y).float()
        
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [ ]:
def evaluate_strategy(pnl):
    # Profitable trades
    wins = pnl[pnl > 0]

    # Losing trades
    losses = pnl[pnl < 0]

    # Performance evaluation metrics
    win_rate = len(wins) / len(pnl) if len(pnl) > 0 else 0
    avg_win = wins.mean() if len(wins) > 0 else 0
    avg_loss = losses.mean() if len(losses) > 0 else 0
    loss_ratio = abs(avg_loss / avg_win) if avg_win != 0 else np.inf
    profit_factor = wins.sum() / abs(losses.sum()) if losses.sum() != 0 else np.inf
    avg_pnl = pnl.mean()
    max_gain = pnl.max()
    max_loss = pnl.min()
    
    # Maximum Drawdown (MDD) calculation
    drawdown = 0
    max_drawdown = 0
    peak = cumulative_pnl[0]
    for val in cumulative_pnl:
        if val > peak:
            peak = val
        drawdown = peak - val
        if drawdown > max_drawdown:
            max_drawdown = drawdown

    # Total trade count
    num_trades = len(pnl)

    # Consolidated evaluation dictionary
    result = {
        "Win Rate": round(win_rate, 4),
        "Average Profit": round(avg_win, 2),
        "Average Loss": round(avg_loss, 2),
        "Risk-Reward Ratio (Avg Loss / Avg Win)": round(loss_ratio, 4),
        "Profit Factor": round(profit_factor, 4),
        "Max Single Profit": round(max_gain, 2),
        "Max Single Loss": round(max_loss, 2),
        "Average PnL per Trade": round(avg_pnl, 2),
        "Max Drawdown": round(max_drawdown, 2),
        "Total Trades": num_trades,
        "Total Net Profit": cumulative_pnl[-1],
        "MAE": mae,
        "MSE": mse,
        "RMSE": rmse,
        "R2 Score": r2
    }
    return result

In [ ]:
# Training Log Management
def save_training_output(base_dir, params: dict, metrics: dict, model, cumulative_pnl):
    log_path = os.path.join(base_dir, "training_log.csv")
    log_entry = {"timestamp": timestamp, **params, **metrics}
    df_entry = pd.DataFrame([log_entry])
    
    # Append to existing log file or create a new one with headers
    if os.path.exists(log_path):
        df_entry.to_csv(log_path, mode='a', header=False, index=False, encoding='utf-8-sig')
    else:
        df_entry.to_csv(log_path, mode='w', header=True, index=False, encoding='utf-8-sig')

    print(f"Saved successfully to: {subdir}")

In [ ]:
for params in grid:
    # Initialize pipeline parameters
    resample_minute = params["resample_minute"]
    random_seed = params["random_seed"]
    window_size = params["window_size"]

    # Initialize model parameters
    dropout_rate = params["dropout_rate"]
    CNN_kernel_size = params["CNN_kernel_size"]
    CNN_1st_channel = params["CNN_1st_channel"]
    CNN_2st_channel = params["CNN_2st_channel"]

    Transformer_encoder_num_layers = params["Transformer_encoder_num_layers"]
    LSTM_hidden_size = params["LSTM_hidden_size"]
    LSTM_num_layers = params["LSTM_num_layers"]

    LSTM_bidirectional = params["LSTM_bidirectional"]
    learning_rate = params["learning_rate"]
    patience = params["patience"]
    num_epochs = params["num_epochs"]
    batch_size = params["batch_size"]

    # Create base directory and generate timestamp
    os.makedirs(base_dir, exist_ok=True)
    timestamp = datetime.now().strftime("%Y-%m-%d %H-%M-%S")

    # Create subdirectory for the current run
    subdir = os.path.join(base_dir, timestamp)
    os.makedirs(subdir, exist_ok=True)

    # Initialize data structures for option data
    data_list = []
    label_list = []

    csv_files = glob.glob(os.path.join(folder_path, "*.csv"))
    print(f"Total CSV files found: {len(csv_files)}")

    for file in csv_files:
        # Load data
        df = pd.read_csv(file)
        df["timestamp"] = pd.to_datetime(df["timestamp"])
        df = df[["timestamp"] + cols_to_keep]

        # Calculate realized volatility for the underlying market (TX)
        df['TX_RV_30'] = df['TX'].diff().pow(2).rolling(30).mean().apply(np.sqrt)
        df['TX_RV_15'] = df['TX'].diff().pow(2).rolling(15).mean().apply(np.sqrt)
        df['TX_RV_10'] = df['TX'].diff().pow(2).rolling(10).mean().apply(np.sqrt)
        df['TX_RV_5'] = df['TX'].diff().pow(2).rolling(5).mean().apply(np.sqrt)

        # Calculate returns percentage change for the underlying market (TX)
        df["TX_pct_change_30"] = df["TX"].pct_change(periods=30)
        df["TX_pct_change_15"] = df["TX"].pct_change(periods=15)
        df["TX_pct_change_10"] = df["TX"].pct_change(periods=10)
        df["TX_pct_change_5"] = df["TX"].pct_change(periods=5)

        # Calculate realized volatility for ATMS
        df['ATMS_RV_30'] = df['ATMS_interpolated'].diff().pow(2).rolling(30).mean().apply(np.sqrt)
        df['ATMS_RV_15'] = df['ATMS_interpolated'].diff().pow(2).rolling(15).mean().apply(np.sqrt)
        df['ATMS_RV_10'] = df['ATMS_interpolated'].diff().pow(2).rolling(10).mean().apply(np.sqrt)
        df['ATMS_RV_5'] = df['ATMS_interpolated'].diff().pow(2).rolling(5).mean().apply(np.sqrt)

        # Calculate percentage change for ATMS
        df["ATMS_pct_change_30"] = df["ATMS_interpolated"].pct_change(periods=30)
        df["ATMS_pct_change_15"] = df["ATMS_interpolated"].pct_change(periods=15)
        df["ATMS_pct_change_10"] = df["ATMS_interpolated"].pct_change(periods=10)
        df["ATMS_pct_change_5"] = df["ATMS_interpolated"].pct_change(periods=5)

        # Calculate candlestick metrics
        df["F_range"] = df["F_high"] - df["F_low"]
        df["F_body"] = df["F_close"] - df["F_open"]

        # Calculate volume moving averages
        df["F_vol_ma_30"] = df["F_vol"].rolling(window=30).mean().fillna(0)
        df["F_vol_ma_15"] = df["F_vol"].rolling(window=15).mean().fillna(0)
        df["F_vol_ma_10"] = df["F_vol"].rolling(window=10).mean().fillna(0)
        df["F_vol_ma_5"] = df["F_vol"].rolling(window=5).mean().fillna(0)

        # Calculate volume percentage changes
        df["F_vol_pct_change_30"] = df["F_vol"].pct_change(periods=30)
        df["F_vol_pct_change_15"] = df["F_vol"].pct_change(periods=15)
        df["F_vol_pct_change_10"] = df["F_vol"].pct_change(periods=10)
        df["F_vol_pct_change_5"] = df["F_vol"].pct_change(periods=5)

        # Calculate time decay differentials
        df["time_diff"] = df["remaining_time"].shift(-1) - df["remaining_time"]

        df = df.dropna().reset_index(drop=True)

        # Configure data resampling rules
        agg_dict = {
            "F_open": "first",
            "F_high": "max",
            "F_low": "min",
            "F_close": "last",
            "F_vol": "sum",
            "TX": "first",
            "ATMS_interpolated": "first",
            "remaining_time": "first"
        }

        for col in df.columns:
            if col != "timestamp" and col not in agg_dict:
                agg_dict[col] = "first"

        df = df.sort_values(by="timestamp")
        df = df.set_index("timestamp")
        df = df.resample(resample_minute).agg(agg_dict)

        # Remove rows with empty bars
        df = df.dropna(subset=["F_open"])

        # Validate that execution volumes and prices are non-zero
        cols_to_check = ["F_open", "F_high", "F_low", "F_close", "F_vol"]
        df = df[(df[cols_to_check] != 0).all(axis=1)]

        # Generate shifting targets for next-step forecasting
        df["next_ATMS"] = df["ATMS_interpolated"].shift(-1)

        # Calculate target percentage changes and corresponding labels
        df["pct_change"] = (df["next_ATMS"] - df["ATMS_interpolated"]) / df["ATMS_interpolated"]
        df["label"] = (df["pct_change"] > 0).astype(int)

        # Drop trailing incomplete record
        df = df.dropna(subset=["next_ATMS"]).reset_index(drop=True)

        # Implement sliding window processing
        for i in range(window_size - 1, len(df)):
            window = df[features].iloc[i - window_size + 1: i + 1].values.astype(np.float32)
            label_value = df[target].iloc[i]
            data_list.append(window)
            label_list.append(label_value)
    
    # Materialize features and labels into NumPy arrays
    X = np.array(data_list)
    X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
    y = np.array(label_list)

    print(f"Total samples generated: {X.shape[0]}, Shape per sample: {X.shape[1:]}")

    # Shuffle datasets for model optimization
    np.random.seed(random_seed)
    indices = np.random.permutation(len(X))

    X_shuffled = X[indices]
    y_shuffled = y[indices]

    # Calculate train / validation / test splits
    num_samples = len(X_shuffled)
    train_size = int(num_samples * 0.8)
    valid_size = int(num_samples * 0.1)
    test_size = num_samples - train_size - valid_size

    # Slice datasets
    X_train = X_shuffled[:train_size]
    y_train = y_shuffled[:train_size]
    X_valid = X_shuffled[train_size:train_size+valid_size]
    y_valid = y_shuffled[train_size:train_size+valid_size]
    X_test  = X_shuffled[train_size+valid_size:]
    y_test  = y_shuffled[train_size+valid_size:]

    # Construct Datasets and DataLoaders
    train_dataset = TimeSeriesDataset(X_train, y_train)
    valid_dataset = TimeSeriesDataset(X_valid, y_valid)
    test_dataset  = TimeSeriesDataset(X_test, y_test)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    valid_loader = DataLoader(valid_dataset, batch_size=batch_size, shuffle=False)
    test_loader  = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    print(f"Train samples: {len(train_dataset)}, Validation samples: {len(valid_dataset)}, Test samples: {len(test_dataset)}")

    # Universal hardware acceleration selection (CUDA / MPS / CPU fallback)
    if torch.cuda.is_available():
        device = torch.device("cuda")
        print("Using GPU:", torch.cuda.get_device_name(device))
    elif torch.backends.mps.is_available():
        device = torch.device("mps")
        print("Using Apple Silicon GPU (MPS)")
    else:
        device = torch.device("cpu")
        print("Using CPU for execution")

    # Instantiate model configuration
    model = FusionNet(window_size=window_size, feature_dim=len(features), dropout_rate=dropout_rate).to(device)
    print(model)

    # Initialize training modules
    criterion = LossFunction()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    early_stop_counter = 0 
    best_valid_loss = float("inf")  
    train_losses = []
    valid_losses = []

    # Model training loop
    for epoch in range(1, num_epochs + 1):
        model.train()
        running_loss = 0.0
        for inputs, targets in train_loader:
            inputs = inputs.to(device)   
            targets = targets.to(device).unsqueeze(1)  
            base_atms = inputs[:, -1, 7]

            optimizer.zero_grad()
            outputs = model(inputs)  
            loss = criterion(outputs, targets, base_atms)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * inputs.size(0)
        
        train_loss = running_loss / len(train_loader.dataset)
        
        # Validation loop
        model.eval()
        valid_running_loss = 0.0
        with torch.no_grad():
            for inputs, targets in valid_loader:
                inputs = inputs.to(device)
                targets = targets.to(device).unsqueeze(1)
                base_atms = inputs[:, -1, 7]

                outputs = model(inputs)
                loss = criterion(outputs, targets, base_atms)
                valid_running_loss += loss.item() * inputs.size(0)
        valid_loss = valid_running_loss / len(valid_loader.dataset)
        
        train_losses.append(train_loss)
        valid_losses.append(valid_loss)
        
        print(f"Epoch {epoch}/{num_epochs} - Train Loss: {train_loss:.4f} - Validation Loss: {valid_loss:.4f}")
        
        # Performance checkpointing and monitoring
        if valid_loss < best_valid_loss:
            best_valid_loss = valid_loss
            torch.save(model.state_dict(), f"{base_dir}/{timestamp}/best_model.pth")
            early_stop_counter = 0 
            print(" ↳ Validation loss decreased, saving model checkpoint.")
        else:
            early_stop_counter += 1
            print(f" ↳ Validation loss did not improve. Early stopping counter: {early_stop_counter} / {patience}")
        
        # Early stopping verification
        if early_stop_counter >= patience:
            print(f" ↳ Validation loss stagnated for {patience} consecutive epochs. Terminating training.")
            break

    # Out-of-sample evaluation setup
    model.load_state_dict(torch.load(f"{base_dir}/{timestamp}/best_model.pth"))
    model.eval()
    all_preds = []
    all_targets = []
    all_base_ATMS = []
    all_TX = []

    # Out-of-sample execution
    with torch.no_grad():
        for inputs, targets in test_loader:
            inputs = inputs.to(device)
            targets = targets.to(device)
            outputs = model(inputs)  
            all_preds.extend(outputs.cpu().numpy().flatten())
            all_targets.extend(targets.cpu().numpy().flatten())
            base = inputs[:, -1, 7].cpu().numpy().flatten()
            all_base_ATMS.extend(base)
            base_TX = inputs[:, -1, 5].cpu().numpy().flatten()
            all_TX.extend(base_TX)

    all_preds = np.array(all_preds)
    all_targets = np.array(all_targets)
    all_base_ATMS = np.array(all_base_ATMS)
    all_TX = np.array(all_TX)

    # Initialize trading PnL arrays
    pnl = np.zeros_like(all_preds)

    # Compute trading performance adjustments
    long_mask = all_preds > all_base_ATMS
    pnl[long_mask] = ((all_targets[long_mask] - all_base_ATMS[long_mask]) / all_base_ATMS[long_mask]) * all_TX[long_mask] / 100
    short_mask = all_preds < all_base_ATMS
    pnl[short_mask] = ((all_base_ATMS[short_mask] - all_targets[short_mask]) / all_base_ATMS[short_mask]) * all_TX[short_mask] / 100

    # Calculate cumulative returns
    cumulative_pnl = np.cumsum(pnl)

    # Generate regression validation metrics
    mse = mean_squared_error(all_targets, all_preds)
    mae = mean_absolute_error(all_targets, all_preds)
    r2 = r2_score(all_targets, all_preds)
    rmse = np.sqrt(mse)

    # Consolidated hyperparameter configurations
    params = {
        "window_size": window_size,
        "batch_size": batch_size,
        "random_seed": random_seed,
        "resample_minute": resample_minute,
        "feature_dim": len(features),
        "learning_rate": learning_rate,
        "MAX_epochs_trained": num_epochs,
        "epochs_trained": epoch,
        "patience": patience,
        "train_data": len(train_dataset),
        "val_data": len(valid_dataset),
        "test_data": len(test_dataset),
        "CNN_kernel_size": CNN_kernel_size,
        "CNN_1st_channel": CNN_1st_channel,
        "CNN_2st_channel": CNN_2st_channel,
        "Transformer_encoder_num_layers": Transformer_encoder_num_layers,
        "LSTM_hidden_size": LSTM_hidden_size,
        "LSTM_num_layers": LSTM_num_layers,
        "LSTM_bidirectional": LSTM_bidirectional,
        "dropout_rate": dropout_rate
    }

    save_training_output(
        base_dir=base_dir,
        params=params,
        metrics=evaluate_strategy(pnl),
        model=model,
        cumulative_pnl=cumulative_pnl
    )

    # Configure Matplotlib styling defaults for international publishing
    plt.rcParams["axes.unicode_minus"] = False     

    # Cumulative PnL visualization
    plt.figure(figsize=(10, 5))
    plt.plot(cumulative_pnl, label="Cumulative PnL", color="blue")
    plt.title("Cumulative PnL Curve Based on Predicted Direction")
    plt.xlabel("Number of Trades")
    plt.ylabel("Cumulative PnL")
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(f"{base_dir}/{timestamp}", "pnl_cumulative.png"))

    # Trade returns distribution histogram
    plt.figure(figsize=(10, 5))
    plt.hist(pnl, bins=100, color='skyblue', edgecolor='black')
    plt.title("Trading PnL Distribution")
    plt.xlabel("PnL per Trade")
    plt.ylabel("Number of Trades")
    plt.grid(True)
    plt.axvline(x=0, color='red', linestyle='--', label="Zero PnL Boundary")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(f"{base_dir}/{timestamp}", "pnl_histogram.png"))

    # Export out-of-sample backtesting metrics
    df_result = pd.DataFrame({
        "TX": all_TX,
        "base_ATMS": all_base_ATMS,
        "true_next_ATMS": all_targets,
        "pred_ATMS": all_preds,
        "pnl": pnl,
        "cumulative_pnl": cumulative_pnl
    })
    df_result.to_csv(f"{base_dir}/{timestamp}/result.csv", index=False, encoding="utf-8-sig")

    # Export execution history losses
    df_loss = pd.DataFrame({
        "train_loss": train_losses,
        "valid_loss": valid_losses
    })
    df_loss.to_csv(f"{base_dir}/{timestamp}/loss.csv", index=False, encoding="utf-8-sig")